# SentinelFlow — 01: Edge Layer (Fast Filter)

**Stack:** ROCm · vLLM · Qwen2.5-VL  
**Layer:** Tier 1 — on-device fast filter  
**Latency budget:** ≤ 10 ms per frame (filter stage), ≤ 30 ms (tiny detector)

### What this notebook does
1. Motion gate — optical flow discards static frames  
2. Perceptual hash — deduplicates near-identical frames  
3. Tiny detector — YOLOv8-nano equivalent (numpy simulation)  
4. **Qwen-VL visual verification** — Qwen2.5-VL describes flagged frames  
5. Adaptive sampler — adjusts FPS based on scene activity  
6. Latency monitor — warns if budget is exceeded

## 0. Load config

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(pathlib.Path('config.yml').read_text())
VLLM_BASE_URL = cfg['vllm_base_url']
QWEN_MODEL    = 'qwen'   # served-model-name set in 00_setup
print('Config loaded:', cfg)

## 1. Imports & Qwen client

In [ ]:
import time, uuid, base64, io, json
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from datetime import datetime

import numpy as np
from PIL import Image
from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-needed')

# ── Latency budget (ms) ──────────────────────────────────────────────────────
BUDGET_MOTION_MS   = 5
BUDGET_HASH_MS     = 2
BUDGET_DETECT_MS   = 30
BUDGET_QWEN_VL_MS  = 2000   # vision LLM is slower — used only on flagged frames

print('✓ Imports done')

## 2. Frame generator (synthetic + real image support)

In [ ]:
def make_synthetic_frames(n: int = 40, h: int = 224, w: int = 224,
                          motion_start: int = 15, motion_len: int = 12) -> List[np.ndarray]:
    """
    Generate synthetic uint8 frames.
    Frames [motion_start : motion_start+motion_len] have high pixel variance (simulated motion).
    """
    rng   = np.random.default_rng(42)
    base  = rng.integers(60, 180, (h, w, 3), dtype=np.uint8)
    frames = []
    for i in range(n):
        scale = 70 if motion_start <= i < motion_start + motion_len else 4
        noise = rng.integers(-scale, scale + 1, base.shape)
        frames.append(np.clip(base.astype(np.int16) + noise, 0, 255).astype(np.uint8))
    return frames

def frame_to_base64(frame: np.ndarray) -> str:
    """Convert numpy HxWx3 uint8 → base64 JPEG string for Qwen-VL."""
    img = Image.fromarray(frame, 'RGB')
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()

FRAMES = make_synthetic_frames(n=40, motion_start=15, motion_len=12)
print(f'Generated {len(FRAMES)} synthetic frames  shape={FRAMES[0].shape}')

## 3. Motion gate (optical flow)

In [ ]:
class MotionGate:
    """
    Dense optical-flow motion gate.
    Production: replace _flow() body with cv2.calcOpticalFlowFarneback().
    """
    def __init__(self, threshold: float = 0.018, budget_ms: float = BUDGET_MOTION_MS):
        self.threshold  = threshold
        self.budget_ms  = budget_ms
        self._prev_gray: Optional[np.ndarray] = None

    def _to_gray(self, frame: np.ndarray) -> np.ndarray:
        return (0.299*frame[:,:,0] + 0.587*frame[:,:,1] + 0.114*frame[:,:,2]).astype(np.float32)

    def _flow(self, prev: np.ndarray, curr: np.ndarray) -> np.ndarray:
        # Production: return cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        diff = curr - prev
        fx   = np.gradient(diff, axis=1)
        fy   = np.gradient(diff, axis=0)
        return np.stack([fx, fy], axis=-1)

    def score(self, frame: np.ndarray) -> float:
        gray = self._to_gray(frame)
        if self._prev_gray is None:
            self._prev_gray = gray
            return 0.0
        flow = self._flow(self._prev_gray, gray)
        self._prev_gray = gray
        mag  = np.sqrt(flow[...,0]**2 + flow[...,1]**2)
        return float((mag > 1.5).mean())

    def should_pass(self, frame: np.ndarray) -> Tuple[bool, float, float]:
        t0     = time.perf_counter()
        s      = self.score(frame)
        ms     = (time.perf_counter() - t0) * 1000
        budget_ok = ms <= self.budget_ms
        if not budget_ok:
            print(f'  ⚠ MotionGate exceeded budget: {ms:.1f}ms > {self.budget_ms}ms')
        return s >= self.threshold, s, ms

gate = MotionGate()
print('MotionGate ready')

## 4. Perceptual hash deduplicator

In [ ]:
class PHashDedup:
    """
    Difference-hash keyframe deduplicator.
    Production: use imagehash.dhash() for a richer 64-bit hash.
    """
    def __init__(self, hamming_threshold: int = 8, budget_ms: float = BUDGET_HASH_MS):
        self.hamming_threshold = hamming_threshold
        self.budget_ms         = budget_ms
        self._last_hash: Optional[int] = None

    def _dhash(self, frame: np.ndarray, size: int = 8) -> int:
        """8x8 difference hash → 64-bit int."""
        # Resize to (size+1) x size
        img   = Image.fromarray(frame).convert('L').resize((size+1, size))
        arr   = np.array(img)
        diff  = arr[:, 1:] > arr[:, :-1]   # horizontal differences
        bits  = diff.flatten().astype(int)
        return int(''.join(map(str, bits)), 2)

    def _hamming(self, a: int, b: int) -> int:
        return bin(a ^ b).count('1')

    def is_new(self, frame: np.ndarray) -> Tuple[bool, str, float]:
        t0 = time.perf_counter()
        h  = self._dhash(frame)
        ms = (time.perf_counter() - t0) * 1000
        if self._last_hash is None:
            self._last_hash = h
            return True, hex(h), ms
        dist = self._hamming(self._last_hash, h)
        if dist >= self.hamming_threshold:
            self._last_hash = h
            return True, hex(h), ms
        return False, hex(h), ms

dedup = PHashDedup()
print('PHashDedup ready')

## 5. Tiny detector (numpy-simulated YOLOv8-nano)

In [ ]:
@dataclass
class Detection:
    event_id   : str
    source_id  : str
    timestamp  : str
    label      : str
    confidence : float
    bbox       : Dict
    motion_score: float
    frame_hash : str

LABELS = ['person', 'vehicle', 'bicycle', 'unknown_object', 'ship', 'wildfire_smoke']

class TinyDetector:
    """
    Simulates YOLOv8-nano inference.

    Production replacement:
        from ultralytics import YOLO
        self.model = YOLO('yolov8n.pt')          # or your domain fine-tuned weights
        self.model.to('cuda')                    # ROCm/CUDA via PyTorch HIP backend
        results = self.model(frame, imgsz=320, verbose=False)
        for box in results[0].boxes:
            ...  # box.xyxyn, box.conf, box.cls
    """
    def __init__(self, min_conf: float = 0.45, source_id: str = 'cam_001',
                 budget_ms: float = BUDGET_DETECT_MS):
        self.min_conf  = min_conf
        self.source_id = source_id
        self.budget_ms = budget_ms

    def detect(self, frame: np.ndarray, motion_score: float,
               frame_hash: str) -> Tuple[List[Detection], float]:
        t0  = time.perf_counter()
        rng = np.random.default_rng(int(frame.flatten()[:4].sum()) % 9999)
        out = []
        for _ in range(int(rng.integers(0, 4))):
            conf = float(rng.uniform(0.35, 0.98))
            if conf < self.min_conf:
                continue
            label = str(rng.choice(LABELS))
            x, y  = float(rng.uniform(0.05, 0.75)), float(rng.uniform(0.05, 0.75))
            w, h_ = float(rng.uniform(0.05, 0.25)), float(rng.uniform(0.05, 0.25))
            out.append(Detection(
                event_id    = str(uuid.uuid4()),
                source_id   = self.source_id,
                timestamp   = datetime.utcnow().isoformat(),
                label       = label,
                confidence  = round(conf, 4),
                bbox        = {'x': round(x,3), 'y': round(y,3),
                               'w': round(w,3), 'h': round(h_,3)},
                motion_score= round(motion_score, 4),
                frame_hash  = frame_hash,
            ))
        ms = (time.perf_counter() - t0) * 1000
        if ms > self.budget_ms:
            print(f'  ⚠ Detector exceeded budget: {ms:.1f}ms > {self.budget_ms}ms')
        return out, ms

detector = TinyDetector()
print('TinyDetector ready')

## 6. Qwen-VL visual verifier (vision-language model)

In [ ]:
class QwenVLVerifier:
    """
    Calls Qwen2.5-VL via vLLM's OpenAI-compatible vision endpoint.
    Only invoked on frames that passed motion+detection gates.
    Returns a structured JSON description and a flag for escalation.
    """
    SYSTEM = (
        'You are a surveillance analysis expert. '
        'Analyse the image and respond ONLY with valid JSON:\n'
        '{"scene": "<1 sentence>", '
        '"objects": ["<obj1>", ...], '
        '"anomaly": true/false, '
        '"severity": "low|medium|high|critical", '
        '"escalate": true/false, '
        '"reason": "<1 sentence>"}'
    )

    def __init__(self, model: str = QWEN_MODEL, budget_ms: float = BUDGET_QWEN_VL_MS):
        self.model     = model
        self.budget_ms = budget_ms

    def verify(self, frame: np.ndarray,
               detections: List[Detection]) -> Tuple[Dict, float]:
        det_hint = ', '.join(f"{d.label}({d.confidence:.2f})" for d in detections)
        prompt   = (
            f'Detector pre-identified: {det_hint or "nothing"}. '
            'Confirm, correct, or add findings.'
        )
        b64 = frame_to_base64(frame)
        t0  = time.perf_counter()

        try:
            resp = client.chat.completions.create(
                model=self.model,
                messages=[{
                    'role': 'system',
                    'content': self.SYSTEM,
                }, {
                    'role': 'user',
                    'content': [
                        {'type': 'image_url',
                         'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
                        {'type': 'text', 'text': prompt},
                    ],
                }],
                max_tokens=256,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            # Strip markdown fences if present
            if raw.startswith('```'):
                raw = '\n'.join(l for l in raw.split('\n')
                                if not l.strip().startswith('```')).strip()
            result = json.loads(raw)
        except Exception as e:
            result = {'scene': 'parse error', 'anomaly': False,
                      'severity': 'low', 'escalate': False,
                      'reason': str(e), 'objects': []}

        ms = (time.perf_counter() - t0) * 1000
        if ms > self.budget_ms:
            print(f'  ⚠ QwenVL exceeded budget: {ms:.0f}ms > {self.budget_ms}ms')
        return result, ms

verifier = QwenVLVerifier()
print('QwenVLVerifier ready')

## 7. Adaptive sampler

In [ ]:
class AdaptiveSampler:
    """Dynamically adjusts capture interval based on scene activity."""
    def __init__(self):
        self.fps_map = {'idle': 1, 'active': 10, 'burst': 30}
        self._interval = 1.0  # seconds
        self._last     = 0.0

    def update(self, motion_score: float, has_detection: bool) -> str:
        if has_detection:
            mode = 'burst'
        elif motion_score > 0.12:
            mode = 'active'
        else:
            mode = 'idle'
        self._interval = 1.0 / self.fps_map[mode]
        return mode

sampler = AdaptiveSampler()
print('AdaptiveSampler ready')

## 8. Full edge pipeline — run all frames

In [ ]:
import warnings
warnings.filterwarnings('ignore')

results = []
stats = dict(received=0, dropped_motion=0, dropped_dup=0,
             processed=0, detections=0, qwen_calls=0)

# Run only first 8 frames through Qwen-VL to keep demo fast
# Remove this cap in production
QWEN_MAX_CALLS = 8

print(f'Processing {len(FRAMES)} frames...\n')
print(f'{"Frame":>5}  {"Motion":>7}  {"Hash":>6}  {"Dets":>4}  {"Mode":>6}  Stage')
print('─' * 60)

for idx, frame in enumerate(FRAMES):
    stats['received'] += 1

    # ── Stage 1: Motion gate ────────────────────────────────────────────────
    passed, mscore, t_motion = gate.should_pass(frame)
    if not passed:
        stats['dropped_motion'] += 1
        mode = sampler.update(mscore, False)
        print(f'{idx:>5}  {mscore:>7.4f}  {"—":>6}  {"—":>4}  {mode:>6}  DROPPED (motion)')
        continue

    # ── Stage 2: Dedup ──────────────────────────────────────────────────────
    is_new, fhash, t_hash = dedup.is_new(frame)
    if not is_new:
        stats['dropped_dup'] += 1
        mode = sampler.update(mscore, False)
        print(f'{idx:>5}  {mscore:>7.4f}  {"dup":>6}  {"—":>4}  {mode:>6}  DROPPED (dup)')
        continue

    # ── Stage 3: Tiny detector ──────────────────────────────────────────────
    stats['processed'] += 1
    dets, t_det = detector.detect(frame, mscore, fhash)
    stats['detections'] += len(dets)

    # ── Stage 4: Qwen-VL on flagged frames (budget-aware) ───────────────────
    vl_result = None
    if dets and stats['qwen_calls'] < QWEN_MAX_CALLS:
        vl_result, t_vl = verifier.verify(frame, dets)
        stats['qwen_calls'] += 1

    # ── Stage 5: Adaptive sampling ──────────────────────────────────────────
    mode = sampler.update(mscore, len(dets) > 0)

    results.append({
        'frame_idx'  : idx,
        'motion_score': mscore,
        'frame_hash' : fhash,
        'detections' : [d.__dict__ for d in dets],
        'qwen_vl'    : vl_result,
        'mode'       : mode,
        'latency_ms' : {'motion': t_motion, 'hash': t_hash, 'detect': t_det},
    })

    det_str = ','.join(f"{d.label[:4]}({d.confidence:.2f})" for d in dets) or 'none'
    print(f'{idx:>5}  {mscore:>7.4f}  {"new":>6}  {len(dets):>4}  {mode:>6}  OK  {det_str}')

print('\n' + '─' * 60)
total = stats['received']
kept  = stats['processed']
print(f'Frames received  : {total}')
print(f'Dropped (motion) : {stats["dropped_motion"]}  ({stats["dropped_motion"]/total*100:.0f}%)')
print(f'Dropped (dup)    : {stats["dropped_dup"]}  ({stats["dropped_dup"]/total*100:.0f}%)')
print(f'Processed        : {kept}   ({kept/total*100:.0f}%)')
print(f'Detections       : {stats["detections"]}')
print(f'Qwen-VL calls    : {stats["qwen_calls"]}')
print(f'Data reduction   : {100*(1-kept/total):.0f}%')

## 9. Inspect Qwen-VL responses

In [ ]:
vl_frames = [r for r in results if r['qwen_vl'] is not None]
print(f'Frames with Qwen-VL analysis: {len(vl_frames)}\n')
for r in vl_frames:
    vl = r['qwen_vl']
    print(f"Frame {r['frame_idx']:>3}  motion={r['motion_score']:.3f}")
    print(f"  Scene    : {vl.get('scene','')}")
    print(f"  Objects  : {vl.get('objects',[])}")
    print(f"  Anomaly  : {vl.get('anomaly')}  Severity: {vl.get('severity')}")
    print(f"  Escalate : {vl.get('escalate')}")
    print(f"  Reason   : {vl.get('reason','')}")
    print()

## 10. Save edge events for Notebook 02

In [ ]:
import json, pathlib

# Flatten all detections into a list of events for the fog layer
edge_events = []
for r in results:
    for det in r['detections']:
        det['qwen_vl']     = r.get('qwen_vl')
        det['mode']        = r['mode']
        det['latency_ms']  = r['latency_ms']
        edge_events.append(det)

pathlib.Path('edge_events.json').write_text(json.dumps(edge_events, default=str, indent=2))
print(f'Saved {len(edge_events)} edge events → edge_events.json')
print('\nSample event:')
if edge_events:
    e = edge_events[0].copy()
    e.pop('qwen_vl', None)   # truncate for display
    print(json.dumps(e, indent=2))